# BookFlix
#### By: Jigisha Tomar

Environment: Python 3 and Jupyter notebook

Libraries used:
* pandas
* re
* nltk
* itertools

## Introduction

The objective of this project is to develop a recommendation system that suggests books to users based on the genres and plotlines of movies they have previously enjoyed. By analyzing a user’s preferences in film genres and narratives, the system will identify and recommend books that align with their tastes, ensuring a personalized and engaging reading experience. A distinctive feature of this system is its commitment to suggesting books that have not been adapted into films. This approach is designed to reduce screen time while fostering a greater interest in reading. By introducing users to compelling literary works beyond the realm of visual media, the system aims to cultivate a deeper appreciation for literature and encourage a habit of reading.

## Importing Libraries

In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import regex as re
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle 
import ast

## Retrieving Data

In [2]:
#Loading datasets
#Movie dataset which contains most on the popular movies
movies= pd.read_csv("data/movie.csv")
movies_df=pd.DataFrame(movies)
movies_df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,en,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,11/5/2014,701729206,169,False,165000000,...,en,Interstellar,The adventures of a group of explorers who mak...,140.241,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,7/16/2008,1004558444,152,False,185000000,...,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,12/15/2009,2923706026,162,False,237000000,...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,4/25/2012,1518815515,143,False,220000000,...,en,The Avengers,When an unexpected enemy emerges and threatens...,98.082,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


In [3]:
movies_df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'budget', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='object')

In [4]:
movies_df.shape

(1048575, 21)

In [5]:
#Books dataset contains books and the detailed decription of that book
books=pd.read_csv("data/books.csv")
books_df=pd.DataFrame(books)
books_df.head(5)

,bookId,title,series,author,rating,description,language,isbn,genres,characters,...,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",...,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,English,9780439358071,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...","['Sirius Black', 'Draco Malfoy', 'Ron Weasley'...",...,06/21/03,['Bram Stoker Award for Works for Young Reader...,2507623,"['1593642', '637516', '222366', '39573', '14526']",98.0,['Hogwarts School of Witchcraft and Wizardry (...,https://i.gr-assets.com/images/S/compressed.ph...,2632233,26923,7.38
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,English,9999999999999,"['Classics', 'Fiction', 'Historical Fiction', ...","['Scout Finch', 'Atticus Finch', 'Jem Finch', ...",...,07/11/60,"['Pulitzer Prize for Fiction (1961)', 'Audie A...",4501075,"['2363896', '1333153', '573280', '149952', '80...",95.0,"['Maycomb, Alabama (United States)']",https://i.gr-assets.com/images/S/compressed.ph...,2269402,23328,NaN
3,1885.Pride_and_Prejudice,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,English,9999999999999,"['Classics', 'Fiction', 'Romance', 'Historical...","['Mr. Bennet', 'Mrs. Bennet', 'Jane Bennet', '...",...,01/28/13,[],2998241,"['1617567', '816659', '373311', '113934', '767...",94.0,"['United Kingdom', 'Derbyshire, England (Unite...",https://i.gr-assets.com/images/S/compressed.ph...,1983116,20452,NaN
4,41865.Twilight,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,English,9780316015844,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...","['Edward Cullen', 'Jacob Black', 'Laurent', 'R...",...,10/05/05,"['Georgia Peach Book Award (2007)', 'Buxtehude...",4964519,"['1751460', '1113682', '1008686', '542017', '5...",78.0,"['Forks, Washington (United States)', 'Phoenix...",https://i.gr-assets.com/images/S/compressed.ph...,1459448,14874,2.1


In [6]:
books_df.shape

(52478, 25)

In [7]:
#Books which are adapted as films
movie_adaptation=pd.read_csv("data/best_movie_adaptations.csv")
movie_adaptation_df=pd.DataFrame(movie_adaptation)
movie_adaptation_df.head()

,Name,Author,Avg Rating,Rating Count,Score,Vote Count
0,The Lord of the Rings (Middle Earth\ #2-4),J.R.R. Tolkien,4.53,693831,8686,89
1,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,4.47,10507368,7453,77
2,Gone with the Wind,Margaret Mitchell,4.31,1236852,4935,50
3,The Godfather (The Godfather\ #1),Mario Puzo,4.39,435479,4674,48
4,To Kill a Mockingbird,Harper Lee,4.26,6382892,4373,45


In [8]:
'''Using regex expression to split the "Name" column to "movie_name" and "movie_parts"
Replacing NaN values in "movie_name" with "Name" column 
as the movie was adapted with the same name that is of book.'''

movie_adaptation_df["movie_name"]=movie_adaptation_df["Name"].str.extract(r'\((.*?)\\')

movie_adaptation_df["movie_parts"]=movie_adaptation_df["Name"].str.extract(r'\#(.-*?)\)')

movie_adaptation_df["Name"] = movie_adaptation_df["Name"].str.replace(r'\(.*?\)', '', regex=True)

movie_adaptation_df["movie_name"]=movie_adaptation_df["movie_name"].fillna(movie_adaptation_df["Name"])

In [9]:
#Displaying the updated Dataframe
movie_adaptation_df.head()

,Name,Author,Avg Rating,Rating Count,Score,Vote Count,movie_name,movie_parts
0,The Lord of the Rings,J.R.R. Tolkien,4.53,693831,8686,89,Middle Earth,NaN
1,Harry Potter and the Sorcerer's Stone,J.K. Rowling,4.47,10507368,7453,77,Harry Potter,1
2,Gone with the Wind,Margaret Mitchell,4.31,1236852,4935,50,Gone with the Wind,NaN
3,The Godfather,Mario Puzo,4.39,435479,4674,48,The Godfather,1
4,To Kill a Mockingbird,Harper Lee,4.26,6382892,4373,45,To Kill a Mockingbird,NaN


In [10]:
movie_adaptation_df.shape

(434, 8)

In [11]:
# Creating a mask for books that are film adaptations
mask = books_df['title'].apply(lambda x: any(name in x for name in movie_adaptation_df['Name']))

# Filter out rows where the mask is True (film adaptations)
books_df = books_df[~mask]

In [12]:
books_df.shape

(51770, 25)

## Data Preparation

#### Movie Dataset

In [13]:
movies_df.dtypes

id                        int64
title                    object
vote_average            float64
vote_count                int64
status                   object
release_date             object
revenue                   int64
runtime                   int64
adult                      bool
budget                    int64
imdb_id                  object
original_language        object
original_title           object
overview                 object
popularity              float64
tagline                  object
genres                   object
production_companies     object
production_countries     object
spoken_languages         object
keywords                 object
dtype: object

In [14]:
movie_dataset=pd.DataFrame(movies_df[["id","title","vote_average","vote_count","popularity","overview","genres","keywords"]])
movie_dataset.set_index("id",inplace=True)
movie_dataset

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,
27205,Inception,8.364,34495,83.952,"Cobb, a skilled thief who commits corporate es...","Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc..."
157336,Interstellar,8.417,32571,140.241,The adventures of a group of explorers who mak...,"Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,..."
155,The Dark Knight,8.512,30619,130.643,Batman raises the stakes in his war on crime. ...,"Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f..."
19995,Avatar,7.573,29815,79.932,"In the 22nd century, a paraplegic Marine is di...","Action, Adventure, Fantasy, Science Fiction","future, society, culture clash, space travel, ..."
24428,The Avengers,7.710,29166,98.082,When an unexpected enemy emerges and threatens...,"Science Fiction, Action, Adventure","new york city, superhero, shield, based on com..."
...,...,...,...,...,...,...,...
905156,鐨勯鏍肩殑椋庢牸儇妒呢蛹刹偎头,0.000,0,0.600,NaN,NaN,NaN
905157,MILF & Cookies 3,0.000,0,0.600,NaN,NaN,NaN
905158,The Choice of Staying,0.000,0,0.600,NaN,Documentary,NaN


In [15]:
movie_dataset[movie_dataset["title"].isna()]

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,
1104880,NaN,0.0,0,0.600,NaN,"Horror, Mystery, Thriller",NaN
1120517,NaN,0.0,0,0.600,NaN,Documentary,NaN
1328667,NaN,0.0,0,0.000,NaN,NaN,NaN
1158567,NaN,0.0,0,0.600,NaN,NaN,NaN
1161361,NaN,0.0,0,0.750,"“My Body, My Rules, and Them” is an exploratio...",NaN,gawad alternatibo
1161605,NaN,0.0,0,0.861,A hitman is tasked to take out ex-mobsters whe...,NaN,NaN
1225818,NaN,0.0,0,0.000,"In this directorial debut of Eden Ewardson, he...",NaN,NaN
518061,NaN,0.0,0,0.600,NONE is a short film that explores the balance...,Animation,NaN
276521,NaN,0.0,0,0.600,NaN,NaN,NaN


In [16]:
movie_dataset=movie_dataset.dropna(subset='title')

In [17]:
movie_dataset[movie_dataset["title"].isna()]

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,


In [18]:
missing_vals=movie_dataset.isna().sum()
missing_vals

title                0
vote_average         0
vote_count           0
popularity           0
overview        215815
genres          414964
keywords        755942
dtype: int64

In [19]:
movie_dataset=movie_dataset.dropna(subset=['genres','overview'])

In [20]:
movie_dataset.isna().sum()

title                0
vote_average         0
vote_count           0
popularity           0
overview             0
genres               0
keywords        314470
dtype: int64

In [21]:
movie_dataset.shape

(530462, 7)

In [22]:
movie_dataset["overview"]=[overview.replace(",","") for overview in movie_dataset["overview"]]

In [23]:
movie_dataset["overview"]=movie_dataset['overview'].apply(lambda x: x.split())

In [24]:
#movie_dataset['overview'] = movie_dataset['overview'].apply(lambda x:[i.replace(",, ",",") for i in x])

In [25]:
movie_dataset.head()

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,
27205,Inception,8.364,34495,83.952,"[Cobb, a, skilled, thief, who, commits, corpor...","Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc..."
157336,Interstellar,8.417,32571,140.241,"[The, adventures, of, a, group, of, explorers,...","Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,..."
155,The Dark Knight,8.512,30619,130.643,"[Batman, raises, the, stakes, in, his, war, on...","Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f..."
19995,Avatar,7.573,29815,79.932,"[In, the, 22nd, century, a, paraplegic, Marine...","Action, Adventure, Fantasy, Science Fiction","future, society, culture clash, space travel, ..."
24428,The Avengers,7.710,29166,98.082,"[When, an, unexpected, enemy, emerges, and, th...","Science Fiction, Action, Adventure","new york city, superhero, shield, based on com..."


In [26]:
movie_dataset['genres'] = [genre.replace(" ", "") for genre in movie_dataset['genres']]

In [27]:
movie_dataset.head()

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,
27205,Inception,8.364,34495,83.952,"[Cobb, a, skilled, thief, who, commits, corpor...","Action,ScienceFiction,Adventure","rescue, mission, dream, airplane, paris, franc..."
157336,Interstellar,8.417,32571,140.241,"[The, adventures, of, a, group, of, explorers,...","Adventure,Drama,ScienceFiction","rescue, future, spacecraft, race against time,..."
155,The Dark Knight,8.512,30619,130.643,"[Batman, raises, the, stakes, in, his, war, on...","Drama,Action,Crime,Thriller","joker, sadism, chaos, secret identity, crime f..."
19995,Avatar,7.573,29815,79.932,"[In, the, 22nd, century, a, paraplegic, Marine...","Action,Adventure,Fantasy,ScienceFiction","future, society, culture clash, space travel, ..."
24428,The Avengers,7.710,29166,98.082,"[When, an, unexpected, enemy, emerges, and, th...","ScienceFiction,Action,Adventure","new york city, superhero, shield, based on com..."


In [28]:
movie_dataset["genres"]= movie_dataset['genres'].astype(str)
movie_dataset["genres"]= movie_dataset['genres'].apply(lambda x: x.split())

In [29]:
movie_dataset.dtypes

title            object
vote_average    float64
vote_count        int64
popularity      float64
overview         object
genres           object
keywords         object
dtype: object

In [30]:
movie_dataset.head()

,title,vote_average,vote_count,popularity,overview,genres,keywords
id,,,,,,,
27205,Inception,8.364,34495,83.952,"[Cobb, a, skilled, thief, who, commits, corpor...","[Action,ScienceFiction,Adventure]","rescue, mission, dream, airplane, paris, franc..."
157336,Interstellar,8.417,32571,140.241,"[The, adventures, of, a, group, of, explorers,...","[Adventure,Drama,ScienceFiction]","rescue, future, spacecraft, race against time,..."
155,The Dark Knight,8.512,30619,130.643,"[Batman, raises, the, stakes, in, his, war, on...","[Drama,Action,Crime,Thriller]","joker, sadism, chaos, secret identity, crime f..."
19995,Avatar,7.573,29815,79.932,"[In, the, 22nd, century, a, paraplegic, Marine...","[Action,Adventure,Fantasy,ScienceFiction]","future, society, culture clash, space travel, ..."
24428,The Avengers,7.710,29166,98.082,"[When, an, unexpected, enemy, emerges, and, th...","[ScienceFiction,Action,Adventure]","new york city, superhero, shield, based on com..."


In [31]:
movie_dataset["tags"]=movie_dataset["overview"]+movie_dataset["genres"]

In [32]:
movie_dataset["tags"]= movie_dataset["tags"].apply(lambda x: " ".join(x))

In [33]:
movie_dataset.head()

,title,vote_average,vote_count,popularity,overview,genres,keywords,tags
id,,,,,,,,
27205,Inception,8.364,34495,83.952,"[Cobb, a, skilled, thief, who, commits, corpor...","[Action,ScienceFiction,Adventure]","rescue, mission, dream, airplane, paris, franc...",Cobb a skilled thief who commits corporate esp...
157336,Interstellar,8.417,32571,140.241,"[The, adventures, of, a, group, of, explorers,...","[Adventure,Drama,ScienceFiction]","rescue, future, spacecraft, race against time,...",The adventures of a group of explorers who mak...
155,The Dark Knight,8.512,30619,130.643,"[Batman, raises, the, stakes, in, his, war, on...","[Drama,Action,Crime,Thriller]","joker, sadism, chaos, secret identity, crime f...",Batman raises the stakes in his war on crime. ...
19995,Avatar,7.573,29815,79.932,"[In, the, 22nd, century, a, paraplegic, Marine...","[Action,Adventure,Fantasy,ScienceFiction]","future, society, culture clash, space travel, ...",In the 22nd century a paraplegic Marine is dis...
24428,The Avengers,7.710,29166,98.082,"[When, an, unexpected, enemy, emerges, and, th...","[ScienceFiction,Action,Adventure]","new york city, superhero, shield, based on com...",When an unexpected enemy emerges and threatens...


In [34]:
movie_dataset["tags"]=movie_dataset["tags"].apply(lambda x: x.lower())

In [35]:
movie_dataset.head()

,title,vote_average,vote_count,popularity,overview,genres,keywords,tags
id,,,,,,,,
27205,Inception,8.364,34495,83.952,"[Cobb, a, skilled, thief, who, commits, corpor...","[Action,ScienceFiction,Adventure]","rescue, mission, dream, airplane, paris, franc...",cobb a skilled thief who commits corporate esp...
157336,Interstellar,8.417,32571,140.241,"[The, adventures, of, a, group, of, explorers,...","[Adventure,Drama,ScienceFiction]","rescue, future, spacecraft, race against time,...",the adventures of a group of explorers who mak...
155,The Dark Knight,8.512,30619,130.643,"[Batman, raises, the, stakes, in, his, war, on...","[Drama,Action,Crime,Thriller]","joker, sadism, chaos, secret identity, crime f...",batman raises the stakes in his war on crime. ...
19995,Avatar,7.573,29815,79.932,"[In, the, 22nd, century, a, paraplegic, Marine...","[Action,Adventure,Fantasy,ScienceFiction]","future, society, culture clash, space travel, ...",in the 22nd century a paraplegic marine is dis...
24428,The Avengers,7.710,29166,98.082,"[When, an, unexpected, enemy, emerges, and, th...","[ScienceFiction,Action,Adventure]","new york city, superhero, shield, based on com...",when an unexpected enemy emerges and threatens...


In [36]:
#key=movie_dataset[movie_dataset["keywords"].isna()]
#key

In [37]:
#tokenizer= RegexpTokenizer(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")
#key["keywords"]=[tokenizer.tokenize(key) for key in key["overview"]]

In [38]:
#key["keywords"]=[[token for token in tokens if len(token)>2] for tokens in key["keywords"]]

In [39]:
#stopwords=stopwords.words("english")

In [40]:
#key["keywords"]=[[token.lower() for token in tokens]for tokens in key["keywords"]]

In [41]:
#key["keywords"]=[[token for token in tokens if token.lower() not in stopwords]for tokens in key["keywords"]]

In [42]:
#Flattening the nested list of tokens
#key["keywords"]=key["keywords"].apply(lambda x:x if isinstance(x,str) else ', ' .join(x))
#key

In [43]:
# Ensure that the 'key' dataframe only has the 'keywords' column
#key = key[['keywords']]

# Replace NaN values in 'keywords' column of movie_dataset with the corresponding values from key['keywords']
#movie_dataset.loc[movie_dataset['keywords'].isna(), 'keywords'] = key['keywords']

# Verify if NaN values are replaced
#print(movie_dataset.isna().sum())


In [44]:
#Ensuring the replacement was successful
#movie_dataset[movie_dataset["title"]=='Ginkgo']

In [45]:
# Define regex pattern to match English titles (letters, numbers, common punctuation)
#pattern = re.compile(r'^[a-zA-Z0-9\s,.\'";!?-]+$')

# Apply regex to filter English titles
#movie_dataset = movie_dataset[movie_dataset["title"].apply(lambda x: bool(pattern.match(str(x))))]


In [46]:
#movie_dataset

#### Book Dataset

In [47]:
books_df.dtypes

bookId               object
title                object
series               object
author               object
rating              float64
description          object
language             object
isbn                 object
genres               object
characters           object
bookFormat           object
edition              object
pages                object
publisher            object
publishDate          object
firstPublishDate     object
awards               object
numRatings            int64
ratingsByStars       object
likedPercent        float64
setting              object
coverImg             object
bbeScore              int64
bbeVotes              int64
price                object
dtype: object

In [48]:
books_dataset=pd.DataFrame(books_df[["bookId","title","series","author","rating","description","genres","awards","coverImg"]])

In [49]:
books_dataset.dtypes

bookId          object
title           object
series          object
author          object
rating         float64
description     object
genres          object
awards          object
coverImg        object
dtype: object

In [50]:
books_dataset.isna().sum()

bookId             0
title              0
series         28557
author             0
rating             0
description     1315
genres             0
awards             0
coverImg         598
dtype: int64

In [51]:
books_dataset[books_dataset["description"].isna()]

,bookId,title,series,author,rating,description,genres,awards,coverImg
291,27494.Leaves_of_Grass,Leaves of Grass,NaN,Walt Whitman,4.12,NaN,"['Poetry', 'Classics', 'Fiction', 'Literature'...",[],https://i.gr-assets.com/images/S/compressed.ph...
680,6295.Howl_and_Other_Poems,Howl and Other Poems,NaN,"Allen Ginsberg, William Carlos Williams (Intro...",4.13,NaN,"['Poetry', 'Classics', 'Fiction', 'American', ...",[],https://i.gr-assets.com/images/S/compressed.ph...
683,323355.The_Book_of_Mormon,The Book of Mormon: Another Testament of Jesus...,NaN,Joseph Smith Jr. (Translator),4.34,NaN,"['Religion', 'Nonfiction', 'Lds', 'Church', 'S...",[],https://i.gr-assets.com/images/S/compressed.ph...
752,1923820.Holy_Bible,Holy Bible: King James Version,NaN,Anonymous,4.41,NaN,"['Religion', 'Classics', 'Nonfiction', 'Christ...",[],https://i.gr-assets.com/images/S/compressed.ph...
846,2666.The_Bonfire_of_the_Vanities,The Bonfire of the Vanities,NaN,Tom Wolfe,3.85,NaN,"['Fiction', 'Classics', 'Novels', 'Literature'...","['Ambassador Book Award for Fiction (1988)', '...",https://i.gr-assets.com/images/S/compressed.ph...
...,...,...,...,...,...,...,...,...,...
52330,6383706-rowlf,Rowlf: Underground 3,Underground,Richard Corben,3.50,NaN,"['Comics', 'Graphic Novels']",[],https://i.gr-assets.com/images/S/compressed.ph...
52340,3457930-refus-global-projections-lib-rantes,Refus Global; Projections Libérantes,NaN,Paul-Emile Borduas,4.00,NaN,[],[],NaN
52407,2823038-forever-young-forever-free-forever-you...,"Forever Young, Forever Free Forever Young, For...",NaN,Hettie Jones,3.67,NaN,[],[],NaN
52420,25313988-spelarens-sten,Spelarens sten,NaN,Bruno K. Öijer,3.60,NaN,['Poetry'],[],https://i.gr-assets.com/images/S/compressed.ph...


In [52]:
#Removing the rows where 'series', 'description' and 'likedPercent' are NaN
books_dataset = books_dataset.dropna(subset=['series', 'description'], how='all')

In [53]:
books_dataset.shape

(50643, 9)

In [54]:
#Now replacing the remaining NaN values
books_dataset.loc[:, 'series'] = books_dataset['series'].fillna('none')
#books_dataset.loc[:,'description']=books_dataset['description'].fillna('not available')

In [55]:
books_dataset=books_dataset.dropna(subset='description')

In [56]:
books_dataset.isna().sum()

bookId           0
title            0
series           0
author           0
rating           0
description      0
genres           0
awards           0
coverImg       378
dtype: int64

In [57]:
books_dataset.shape

(50455, 9)

In [58]:
books_dataset.head()

,bookId,title,series,author,rating,description,genres,awards,coverImg
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...",['Locus Award Nominee for Best Young Adult Boo...,https://i.gr-assets.com/images/S/compressed.ph...
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",['Bram Stoker Award for Works for Young Reader...,https://i.gr-assets.com/images/S/compressed.ph...
5,19063.The_Book_Thief,The Book Thief,none,Markus Zusak (Goodreads Author),4.37,Librarian's note: An alternate cover edition c...,"['Historical Fiction', 'Fiction', 'Young Adult...",['National Jewish Book Award for Children’s an...,https://i.gr-assets.com/images/S/compressed.ph...
6,170448.Animal_Farm,Animal Farm,none,"George Orwell, Russell Baker (Preface), C.M. W...",3.95,Librarian's note: There is an Alternate Cover ...,"['Classics', 'Fiction', 'Dystopia', 'Fantasy',...","['Prometheus Hall of Fame Award (2011)', 'Retr...",https://i.gr-assets.com/images/S/compressed.ph...
7,11127.The_Chronicles_of_Narnia,The Chronicles of Narnia,The Chronicles of Narnia (Publication Order) #1–7,"C.S. Lewis, Pauline Baynes (Illustrator)",4.26,"Journeys to the end of the world, fantastic cr...","['Fantasy', 'Classics', 'Fiction', 'Young Adul...",[],https://i.gr-assets.com/images/S/compressed.ph...


In [59]:
# defining a function to process text: remove special characters & split multi-word phrases
def process_text(text):
    if isinstance(text, list):  # Handle list format in 'genres' column
        text = ', '.join(text)  
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove special characters
    words = text.split()  # Split into individual words
    return ', '.join(words)  

# Apply processing to both columns
books_dataset['description'] = books_dataset['description'].apply(process_text)
books_dataset['genres'] = books_dataset['genres'].apply(process_text)


In [60]:
books_dataset['tags'] = books_dataset['description'] + books_dataset['genres']

In [61]:
books_dataset["tags"]=books_dataset["tags"].apply(lambda x: x.lower())

In [62]:
books_dataset.head()

,bookId,title,series,author,rating,description,genres,awards,coverImg,tags
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,"WINNING, MEANS, FAME, AND, FORTUNELOSING, MEAN...","Young, Adult, Fiction, Dystopia, Fantasy, Scie...",['Locus Award Nominee for Best Young Adult Boo...,https://i.gr-assets.com/images/S/compressed.ph...,"winning, means, fame, and, fortunelosing, mean..."
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,"There, is, a, door, at, the, end, of, a, silen...","Fantasy, Young, Adult, Fiction, Magic, Childre...",['Bram Stoker Award for Works for Young Reader...,https://i.gr-assets.com/images/S/compressed.ph...,"there, is, a, door, at, the, end, of, a, silen..."
5,19063.The_Book_Thief,The Book Thief,none,Markus Zusak (Goodreads Author),4.37,"Librarians, note, An, alternate, cover, editio...","Historical, Fiction, Fiction, Young, Adult, Hi...",['National Jewish Book Award for Children’s an...,https://i.gr-assets.com/images/S/compressed.ph...,"librarians, note, an, alternate, cover, editio..."
6,170448.Animal_Farm,Animal Farm,none,"George Orwell, Russell Baker (Preface), C.M. W...",3.95,"Librarians, note, There, is, an, Alternate, Co...","Classics, Fiction, Dystopia, Fantasy, Literatu...","['Prometheus Hall of Fame Award (2011)', 'Retr...",https://i.gr-assets.com/images/S/compressed.ph...,"librarians, note, there, is, an, alternate, co..."
7,11127.The_Chronicles_of_Narnia,The Chronicles of Narnia,The Chronicles of Narnia (Publication Order) #1–7,"C.S. Lewis, Pauline Baynes (Illustrator)",4.26,"Journeys, to, the, end, of, the, world, fantas...","Fantasy, Classics, Fiction, Young, Adult, Chil...",[],https://i.gr-assets.com/images/S/compressed.ph...,"journeys, to, the, end, of, the, world, fantas..."


### Removing Stopwords

In [63]:
stop_words=stopwords.words("english")

In [64]:
def remove_stopwords(text):
    if isinstance(text, str):  # Ensure the value is a string
        words = text.split()
        filtered_words = [word for word in words if word.lower() not in stop_words]
        return ' '.join(filtered_words)
    return text  # Return as is if not a string



In [65]:
movie_dataset['tags'] = movie_dataset['tags'].apply(remove_stopwords)


In [66]:
books_dataset['tags'] = books_dataset['tags'].apply(remove_stopwords)


## Data Modelling 

In [67]:
# Combine genres and text for better recommendations
#movie_dataset["content"] = movie_dataset["genres"] + " " + movie_dataset["overview"]]]]]
#books_dataset["content"] = books_dataset["genres"] + " " + books_dataset["description"]

In [68]:
# Vectorize the text using TF-IDF
vectorizer = TfidfVectorizer()
movie_vectors = vectorizer.fit_transform(movie_dataset["tags"])
book_vectors = vectorizer.transform(books_dataset["tags"])

In [69]:
print(movie_vectors)

  (0, 54863)	0.24049123557264313
  (0, 248779)	0.19982531072409496
  (0, 269976)	0.1645940990705232
  (0, 56354)	0.1868851432969011
  (0, 59164)	0.18354934366512876
  (0, 84871)	0.2058568733237489
  (0, 125906)	0.24147445999593836
  (0, 259000)	0.4295305738085157
  (0, 265957)	0.19898956123954104
  (0, 193690)	0.17898418978660532
  (0, 48829)	0.13604298284478117
  (0, 223234)	0.18473112739745615
  (0, 194491)	0.09216375582299491
  (0, 156124)	0.07472557863387887
  (0, 203114)	0.2162663493096297
  (0, 266228)	0.16572434892585206
  (0, 57750)	0.16589017101495268
  (0, 124667)	0.16667405040509697
  (0, 125047)	0.2261056749521505
  (0, 124584)	0.2987129736474226
  (0, 15110)	0.11772717442422012
  (0, 205143)	0.13562783836831865
  (0, 122861)	0.15052955761224934
  (0, 265953)	0.17207104111715785
  (0, 6516)	0.08557918618596244
  :	:
  (530460, 91534)	0.08486439652122882
  (530460, 75139)	0.23073883739823947
  (530460, 248243)	0.10467552340611372
  (530460, 57363)	0.11411268819677077
  (5304

In [70]:
print(book_vectors)

  (0, 6516)	0.029063115509465734
  (0, 7362)	0.053840992169616714
  (0, 7426)	0.03402475113600129
  (0, 8119)	0.1599967152100758
  (0, 8249)	0.06061870207227872
  (0, 11160)	0.04646834320075462
  (0, 12758)	0.04601243257713819
  (0, 13769)	0.28084780249262703
  (0, 15014)	0.05715151187298503
  (0, 16160)	0.06278018082645553
  (0, 18944)	0.1262447166995865
  (0, 28259)	0.03769565797147753
  (0, 28461)	0.0729734744460717
  (0, 28674)	0.0652778286810414
  (0, 31101)	0.06996299114458766
  (0, 37814)	0.038537981445972924
  (0, 42325)	0.12082921490912818
  (0, 42591)	0.11806561197318248
  (0, 45088)	0.14970449167573124
  (0, 48124)	0.05521462862274363
  (0, 51764)	0.06131900142434183
  (0, 54456)	0.046458974922727214
  (0, 58079)	0.07741500818438295
  (0, 61595)	0.0596068525620122
  (0, 65832)	0.04296672560229094
  :	:
  (50454, 182595)	0.0637239228602926
  (50454, 197737)	0.08139201055385176
  (50454, 201768)	0.07084250855065717
  (50454, 214690)	0.17832855962058664
  (50454, 216505)	0.0791

In [71]:
def recommend_books(movie_title, top_n=5):
    # Convert both input and dataset titles to lowercase for case-insensitive matching
    movie_title = movie_title.lower()
    movie_dataset["title_lower"] = movie_dataset["title"].str.lower()
    
    if movie_title not in movie_dataset["title_lower"].values:
        return "Movie not found. Please enter a valid movie title."
    
    movie_index = movie_dataset[movie_dataset["title_lower"] == movie_title].index[0]
    movie_vector = movie_vectors[movie_index]
    
    similarity_scores = cosine_similarity(movie_vector.reshape(1, -1), book_vectors)[0]
    
    top_indices = np.argsort(similarity_scores)[-top_n:][::-1]
    
    recommendations = []
    
    for index in top_indices:
        book_title = books_dataset.iloc[index]["title"]
        author = books_dataset.iloc[index]["author"]
        awards = books_dataset.iloc[index]["awards"]
        rating = books_dataset.iloc[index]["rating"]
        
        # Construct book dictionary
        book_info = {
            "title": book_title,
            "author": author,
            "ratings": rating
        }
        
        # Only include awards if they are not an empty list
        if isinstance(awards, list) and awards:  # Ensuring it's a list and not empty
            book_info["awards"] = awards
        
        recommendations.append(book_info)
    
    return recommendations


In [72]:
# Print movies
user_movie = "Inception"  # Replace this with user input
recommended_books = recommend_books(user_movie)
print(f"Books recommended for fans of {user_movie}: {recommended_books}")

Books recommended for fans of Inception: [{'title': 'One Real Thing', 'author': 'Anah Crow (Goodreads Author), Dianne Fox', 'ratings': 3.77}, {'title': 'Phoenix Rising', 'author': 'JoLynne Valerie', 'ratings': 4.24}, {'title': 'My Map Of You', 'author': 'Isabelle Broom (Goodreads Author)', 'ratings': 4.06}, {'title': 'The Legend of Holly Claus', 'author': 'Brittney Ryan (Goodreads Author), Laurel Long (Illustrator)', 'ratings': 4.23}, {'title': 'A Proscriptive Relationship', 'author': 'Jordan Lynde (Goodreads Author)', 'ratings': 4.28}]


In [73]:
import pickle

# Dictionary to save everything needed for recommendations
model_data = {
    "movie_vectors": movie_vectors,  # Numpy array of movie embeddings
    "book_vectors": book_vectors,    # Numpy array of book embeddings
    "movie_dataset": movie_dataset,  # Movie metadata
    "books_dataset": books_dataset,  # Book metadata
}

# Save the model as a pickle file
with open("book_recommendation_model.pkl", "wb") as file:
    pickle.dump(model_data, file)

print("Model saved successfully as 'book_recommendation_model.pkl'")


Model saved successfully as 'book_recommendation_model.pkl'
